# Protein ↔ Phenotype Relation-Wise Merge

Merges Protein–Phenotype triples from CrossBAR;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [12]:
import os
import pandas as pd

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PROTEIN_PHENOTYPE/ALL_PROTEIN_PHENOTYPE.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Load KG Sources

### 1a. CrossBAR

In [4]:
CrossBAR_Protein_Phenotype = pd.read_csv(PROC_DIR + 'crossbar/Protein_Phenotype_Crossbar.csv')
CrossBAR_Protein_Phenotype.columns = CrossBAR_Protein_Phenotype.columns.str.lower()
print(f"CrossBAR: {len(CrossBAR_Protein_Phenotype):,} rows | columns: {list(CrossBAR_Protein_Phenotype.columns)}")
CrossBAR_Protein_Phenotype['kg_type'] = 'Generalised'
CrossBAR_Protein_Phenotype['kg_source'] = 'crossbar'
CrossBAR_Protein_Phenotype['species'] = 'Homo species'

CrossBAR_Protein_Phenotype.head(2)

CrossBAR: 328 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_alt_id', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,F7VJQ1,Protein_Phenotype,HP:0040201,Protein,Phenotype,crossbar,PRNP,Alternative prion protein,Simultanapraxia,Uniprot_ID,HPO_ID,Generalised,Homo species
1,F7VJQ1,Protein_Phenotype,HP:0040200,Protein,Phenotype,crossbar,PRNP,Alternative prion protein,Motor impersistence,Uniprot_ID,HPO_ID,Generalised,Homo species


## 2. Check and Fix Duplicate Columns

In [5]:
SOURCE_DFS = [('CrossBAR_Protein_Phenotype', CrossBAR_Protein_Phenotype)]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[CrossBAR_Protein_Phenotype] ✓ no duplicates


## 3. Align Schemas and Concatenate

In [6]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 328 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,F7VJQ1,Protein_Phenotype,HP:0040201,Protein,<NA>,Phenotype,crossbar,Uniprot_ID,HPO_ID,Alternative prion protein,Simultanapraxia,Generalised,Homo species
1,F7VJQ1,Protein_Phenotype,HP:0040200,Protein,<NA>,Phenotype,crossbar,Uniprot_ID,HPO_ID,Alternative prion protein,Motor impersistence,Generalised,Homo species


## 4. Standardise Fixed-Value Columns

In [7]:
final_df['head_type'] = 'Protein'
final_df['tail_type'] = 'Phenotype'
final_df['relation']  = 'Protein_Phenotype'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Protein_Phenotype'}
Unique head_id_is: {'Uniprot_ID'}
Unique tail_id_is: {'HPO_ID'}
Unique kg_source: {'crossbar'}


## 5. Deduplicate and Add Schema Columns

In [8]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"After deduplication: {len(final_df_dedup):,} rows")
final_df_dedup.head()

After deduplication: 328 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A0A1W2PR82,Protein_Phenotype,HP:0002014,Protein,None,Phenotype,crossbar,Uniprot_ID,HPO_ID,Protein PERCC1 {ECO:0000305},Diarrhea,Generalised,Homo species
1,A0A1W2PR82,Protein_Phenotype,HP:0002242,Protein,None,Phenotype,crossbar,Uniprot_ID,HPO_ID,Protein PERCC1 {ECO:0000305},Abnormal intestine morphology,Generalised,Homo species
2,A0A1W2PR82,Protein_Phenotype,HP:0002244,Protein,None,Phenotype,crossbar,Uniprot_ID,HPO_ID,Protein PERCC1 {ECO:0000305},Abnormality of the small intestine,Generalised,Homo species
3,A0A1W2PR82,Protein_Phenotype,HP:0011024,Protein,None,Phenotype,crossbar,Uniprot_ID,HPO_ID,Protein PERCC1 {ECO:0000305},Abnormality of the gastrointestinal tract,Generalised,Homo species
4,A0A1W2PR82,Protein_Phenotype,HP:0011458,Protein,None,Phenotype,crossbar,Uniprot_ID,HPO_ID,Protein PERCC1 {ECO:0000305},Abdominal symptom,Generalised,Homo species


## 6. QC — NaN Check

In [9]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,328,0,328
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 7. Save Output

In [13]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 328 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PROTEIN_PHENOTYPE/ALL_PROTEIN_PHENOTYPE.csv
